# **AI Agent Tutorial - Study Planner Agent**
> Study Planner Agent creates personalized study schedules based on user input and deadlines.
* Tasks are automatically generated and pushed directly to Jira as issues for easy tracking and execution.
* The Study Planner Agent Extension can book calendar reminders to help you stay on track.

## **0. Install Dependencies**

In [ ]:
!pip install Phidata Groq youtube_transcript_api dotenv jira pycountry

# firecrawl langchain openai langchain_openai langchain_community pyngrok

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 716.9/716.9 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.2/130.2 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 485.7/485.7 kB 19.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.5/77.5 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 63.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 236.0/236.0 kB 13.7 MB/s eta 0:00:00


## **1. Setup Initialization**
### **1.1. [Pre-requisite] Get API key for Groq**

Step 1: Visit the Groq Developer Portal

* Open your browser and go to: https://console.groq.com

Step 2: Sign Up or Log In
* If you already have an account, click Log In.
* If you’re new, click Sign Up and follow the prompts to create an account (you may need to verify your email).

Step 3: Access the API Section
* Once logged in, you'll land on the Groq Console.
* Navigate to the API Keys section from the sidebar or dashboard.

Step 4: Generate a New API Key
* Click the “Create API Key” button.
* Give your key a name (e.g., "workshop-key").
* Click Create or Generate.

Step 5: Copy and Store the Key Securely
* Your API key will be shown only once — copy it immediately and store it in a secured location.
* Never expose your API key in client-side code or public repositories.

### **1.2. Add the API key in the Secret Manager**
Step 1: Click on Secrets (Key sign) on the left pane of colab

Step 2: Provide the name as ```GROQ_API_KEY``` and Value as API Key copied in Step 5 of 1.1

Step 3: Toggle "ON" the notebook access.

## **2. Setup Jira Account**
### **2.1. [Pre-requisite] Get API Key for Jira**
Step 1: Sign up/Log in to the Jira account
* Sign up on the website - https://www.atlassian.com/software/jira using your email address, Google, or Microsoft account.

Step 2: How to get Jira API Key
* Go to https://id.atlassian.com/manage-profile/security/api-tokens page
* Click on Create Classic API Token.
* Give a name like "Test API"
* Copy and store the key securely.

Step 3: Jira Server URL
* The Jira Server URL is your Atlassian site URL, usually ending in `.atlassian.net`.

Step 4: Jira Username
* We need to set JIRA_USERNAME environment variable with the email address that is used to sign up for the Jira account.

Step 5: Add the following to the Secret Manager
* Enter `JIRA_API_KEY` as the name and the `<API KEY>` as value
* Enter `JIRA_SERVER_URL` as the name and add `<Server URL>` (like `https://xxx.atlassian.net`) as value
* Enter `JIRA_USERNAME` as the name and add `<EMAIL ID>` used to sign up as the value.



### **2.2. Create a project**
* In the project templates pane, choose Software Development, then on the main window choose Kanban
* Select a team managed project
* On the "Add project details" page, set the name as "Test Project", set the key to "TES" (*This is important*) and set the access to Open.

## **3. Setup Cal.com Account**
### **3.1. [Pre-requisite] Get API Key for Cal.com**
Step 1: Create an account on Cal.com
* Setup an account on Cal.com if it does not exist
* In the availability on the left pane, keep all slots open (note if the slot is not open, we cannot create a meeting invite).

Step 2: Get API Key
* Once logged in, go to https://app.cal.com/settings/developer/api-keys
* Click “+ Add”
* Give it a name - "Test AI Agent"
* Copy and store the key securely.

Step 3: Get Event type id
* Run on `curl.exe -H "Authorization: Bearer <CALCOM_API_KEY>" https://api.cal.com/v2/event-types` on powershell (windows).  If you are using Mac, use "curl" instead of "curl.exe".
* Event type id is the value of `id` in the `eventTypes` element

Step 4: Add the following to the Secret Manager
* Enter `CALCOM_API_KEY` as the name and the `<API KEY>` as value
* Enter `CALCOM_EVENT_TYPE_ID` as the name and add the `id` from Step 3 above
* Enter `EMAIL_ID` as the name and add `<EMAIL ID>` to which the calendar invitations need to be sent.

In [ ]:
import os
from google.colab import userdata

os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')                   ## follow step 4 of section 1.1
os.environ["JIRA_TOKEN"] = userdata.get('JIRA_API_KEY')                     ## follow step 2 of section 2.1
os.environ["JIRA_SERVER_URL"] = userdata.get('JIRA_SERVER_URL')             ## follow step 3 of section 2.1
os.environ["JIRA_USERNAME"] = userdata.get('JIRA_USERNAME')                 ## follow step 4 of section 2.1
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
os.environ["CALCOM_API_KEY"] = userdata.get('CALCOM_API_KEY')               ## follow step 2 of section 3.1
os.environ["CALCOM_EVENT_TYPE_ID"] = userdata.get('CALCOM_EVENT_TYPE_ID')   ## follow step 3 of section 3.1

In [ ]:
from phi.agent import Agent
from phi.model.openai import OpenAIChat
from phi.tools.jira_tools import JiraTools
from phi.model.groq import Groq
from phi.tools.calcom import CalCom
from datetime import datetime

study_partner = Agent(
    name="Study Planner Agent",
   model=OpenAIChat(id="gpt-4o"),
    # model=Groq(id="llama-3.3-70b-versatile"),  ## Toggle with different LLM model
    tools=[JiraTools(), CalCom()],
    markdown=True,
    description="You are a study partner who assists users in finding resources, answering questions, and providing explanations on various topics.",
    instructions=[
        "Search for relevant information on the given topic and verify information from multiple reliable sources.",
        "Break down complex topics into digestible chunks and provide step-by-step explanations with practical examples.",
        "Share curated learning resources including documentation, tutorials, articles, research papers, and community discussions.",
        "Recommend high-quality YouTube videos and online courses that match the user's learning style and proficiency level.",
        "Suggest hands-on projects and exercises to reinforce learning, ranging from beginner to advanced difficulty.",
        "Create personalized study plans with clear milestones, deadlines, and progress tracking.",
        "Provide tips for effective learning techniques, time management, and maintaining motivation.",
        "Recommend relevant communities, forums, and study groups for peer learning and networking.",
        f"You can also help in scheduling meetings so it does not slip the calendars. Today is {datetime.now()}."
    ],
)
project_id = "TES"  ## to be updated if a different project id is selected in section 2.2
email_id = userdata.get('EMAIL_ID') ## Sends meeting invite from cal.com to this email

study_partner.print_response(
    f"""I want to learn about machine learning in depth. I know the basics, have 2 weeks to learn, and can spend 2 hours daily.
    Please create up to 1 task per day in Jira in project {project_id} with summary and description on how to gradually improve my machine learning knowledge.
    Please create bookings with {email_id} each day for the next two weeks at 9pm pst with summary as the subject line and description.
    Please create the tasks and make the booking directly without asking for any confirmation""",
    stream=True)


Output()

# **4. Appendix**

### **4.1. Get API Key for Open AI (optional)**
Here are the instructions to get API Key for OpenAI (**Note that this requires purchase**)

Step 1: Sign In to OpenAI
* Go to https://platform.openai.com
* Click “Sign in” or “Sign up” if you don’t have an account

Step 2: Navigate to API Keys
* Once logged in, click your profile icon (top-right corner)
* Select “View API Keys” from the dropdown
* Or visit directly: https://platform.openai.com/account/api-keys

Step 3: Create a New API Key
* Click “+ Create new secret key”
* Enter a name for your key (e.g., "Testing Agent")
* Click Create secret key
* Copy the key immediately – it will not be shown again for security reasons

Step 4: Add it in the Colab Secret Manager with `OPENAI_API_KEY` as the name

(**Note this step requires purchase.  It is not mandatory for the workshop**)

Step 5 : Buy credits to use the API here - https://platform.openai.com/settings/organization/billing/overview

### **4.2. Different models in Groq (optional)**
Here are the a few models in Groq.  More details here - https://console.groq.com/docs/models
1. gemma2-9b-it
2. llama-3.3-70b-versatile
3. llama-3.1-8b-instant
4. llama3-70b-8192